[env: optonpx]

# 2.2 (SpikeInterface variant) — processed sessions for spike sorting

Same result as `2.2_concatenate_sessions.ipynb`, but the signal processing and the
concatenation are done with **SpikeInterface** instead of `src.preprocess`:

| step | numpy version | this version |
|---|---|---|
| laser pulse detection (ADC) | `src.detect_laser_pulses` | `src.detect_laser_pulses` (unchanged) |
| ADC→probe lag estimation | `src.estimate_artifact_lag` | `src.estimate_artifact_lag` (unchanged) |
| artifact mask | `apply_ramp_mask` (linear ramp) | `spikeinterface.preprocessing.remove_artifacts(mode="linear")` |
| NP2.0 phase shift | `phase_shift_chunk` (FFT) | `spikeinterface.preprocessing.phase_shift` |
| concatenate + write int16 | `write_concatenated_recording` | `concatenate_recordings` + `write_binary_recording` |

Per session, in order: **mask** (optotagging sessions only) → **phase shift** →
append. `phase_shift` runs per session *before* `concatenate_recordings`, so an
FFT block never crosses a session boundary. CAR is **not** done here.

**Raw recordings are never modified** — SpikeInterface opens them read-only and
all output goes under `processed_session`.

# 2.2 Concatenate several ephys data sections of a single session

Typically concatenate the behavior and the optotagging section of a single session.

If you have different session structure to process, it's also easy to make modification based on this notebook.

## Steps

Included:

1. mask optoelectric artifact
2. mask discrete artifact (generated by touching water spout + poor grounding) (TODO)
3. phase shift
4. concatenate

Optional:

1. filter (off by default, included in kilosort) (TODO)
2. common average reference (off by default, included in kilosort)

In [4]:
from pathlib import Path
import yaml
from pprint import pprint

# ========== Specify the session to preprocess ==========
session_name = "IM-1979_2026-09-11"
base_dir_raw = Path(r"E:/")
base_dir_output = Path(r"E:/")
is_overwrite = False
# ========================================================

# Load json file
metadata_dir = Path.cwd().parent
metadata_path = Path(metadata_dir / "metadata" / f"{session_name}.yaml")
print(metadata_path)
with open(metadata_path) as f:
    config = yaml.safe_load(f)

base_dir_raw = base_dir_raw / config["animal"]
behavior_session = base_dir_raw / "ephys" / config["sessions"]["behavior"]
tagging_session = base_dir_raw / "ephys" / config["sessions"]["optotagging"]
processed_session = base_dir_output / config["animal"] / "ephys" / config["sessions"]["processed"]

probe_names = config["streams"]["probes"]
n_probe = len(probe_names)
print(f"There are {len(probe_names)} probes: {', '.join(probe_names)}")

if processed_session.exists() and not is_overwrite:
    raise ValueError("Processed folder already existed and 'overwrite' is False.")

c:\Users\Jeff\Files\Code\optonpx\metadata\IM-1979_2026-09-11.yaml
There are 2 probes: probeA, probeB


ValueError: Processed folder already existed and 'overwrite' is False.

In [6]:
import sys, json, csv
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
from src import (
    oe_parse_folders, oe_parse_params, oe_load_adc_column,
    save_kilosort_channel_map, detect_laser_pulses, estimate_artifact_lag,
)
from src.preprocess import (
    npx2_sample_shifts,
    DEFAULT_PULSE_DETECTION, DEFAULT_LAG_ESTIMATION, DEFAULT_ARTIFACT_MASK,
)

import spikeinterface.full as si
from spikeinterface.preprocessing import phase_shift, remove_artifacts
from spikeinterface.core import concatenate_recordings, write_binary_recording
print("spikeinterface", si.__version__)

# Per-probe parameter blocks from the metadata (defaults filled in from src.preprocess)
default_phase_shift_params = {"apply": True}                       # + margin_ms from metadata
default_mask_laser_params  = {**DEFAULT_ARTIFACT_MASK, "apply": True}
default_car_params         = {"apply": False}                      # KS4 does CAR, per shank

concatenate_order  = {}
phase_shift_params = {}
mask_laser_params  = {}
car_params         = {}
for probe_name in probe_names:
    preprocess_cfg = config[probe_name]["preprocess"]
    concatenate_order[probe_name]  = preprocess_cfg["concatenate"]
    phase_shift_params[probe_name] = {**default_phase_shift_params, **preprocess_cfg["phase_shift"]}
    mask_laser_params[probe_name]  = {**default_mask_laser_params,  **preprocess_cfg["mask_laser_artifact"]}
    car_params[probe_name]         = {**default_car_params,         **preprocess_cfg["car"]}

spikeinterface 0.104.5


In [8]:
# --- resolve session folders, parse Open Ephys, check they can be concatenated ---
session_dirs  = {"behavior": behavior_session, "optotagging": tagging_session}
session_roles = list(session_dirs)                           # ["behavior", "optotagging"]


def match_one_stream(items, stream_name, name_of):
    """Return the single item whose name ends with `stream_name`, case-insensitively
    (the metadata says 'probeA', Open Ephys says 'ProbeA')."""
    matches = [item for item in items
               if str(name_of(item)).split(".")[-1].lower() == stream_name.lower()]
    if len(matches) != 1:
        raise ValueError(f"probe {stream_name!r}: expected 1 match, got {[name_of(i) for i in items]}")
    return matches[0]


sessions = {}   # role -> {folder, adc_dir, adc_params, probes: {probe_name: {...}}}
for role, folder in session_dirs.items():
    _, oe_paths = oe_parse_folders(folder)
    probe_params_list, adc_params = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])
    session_info = {"folder": folder, "adc_dir": oe_paths["adc_stream"],
                    "adc_params": adc_params, "probes": {}}
    for probe_name in probe_names:
        probe_params = match_one_stream(probe_params_list, probe_name, lambda p: p["stream_name"])
        stream_dir   = match_one_stream(oe_paths["ephys_streams"], probe_name, lambda p: p.name)
        n_samples    = int(len(np.load(stream_dir / "timestamps.npy", mmap_mode="r")))
        session_info["probes"][probe_name] = {
            "params": probe_params, "stream_dir": stream_dir,
            "n_chan": int(probe_params["channel_count"]),
            "fs": float(probe_params["sample_rate"]), "n_samples": n_samples}
    sessions[role] = session_info

# concatenating only makes sense if every session used the exact same probe wiring
reference_role = session_roles[0]
for probe_name in probe_names:
    reference_params = sessions[reference_role]["probes"][probe_name]["params"]
    for role in session_roles[1:]:
        params = sessions[role]["probes"][probe_name]["params"]
        for field in ("stream_name", "channel_count", "sample_rate"):
            if str(params[field]) != str(reference_params[field]):
                raise ValueError(f"{probe_name}: {field} differs "
                                 f"({reference_role}={reference_params[field]!r} vs {role}={params[field]!r})")
        for field in ("channel_bank", "channel_shank", "channel_electrode"):
            for channel in range(int(reference_params["channel_count"])):
                if reference_params[field].get(channel) != params[field].get(channel):
                    raise ValueError(f"{probe_name} ch{channel} {field} differs between "
                                     f"{reference_role} and {role} — different probe/bank config, "
                                     "cannot concatenate")

print("compatibility OK: stream / channel_count / sample_rate / per-channel bank+shank+electrode match")
for probe_name in probe_names:
    sample_counts = [sessions[role]["probes"][probe_name]["n_samples"] for role in session_roles]
    fs = sessions[reference_role]["probes"][probe_name]["fs"]
    print(f"  {probe_name}: {sample_counts} -> {sum(sample_counts)} samples ({sum(sample_counts) / fs:.1f} s)")

compatibility OK: stream / channel_count / sample_rate / per-channel bank+shank+electrode match
  probeA: [168631385, 30657261] -> 199288646 samples (6643.0 s)
  probeB: [168631100, 30657248] -> 199288348 samples (6642.9 s)


In [ ]:
# --- safety: the processed output must not sit inside any raw recording folder ---
output_folder = processed_session.resolve()
for role, folder in session_dirs.items():
    raw_folder = folder.resolve()
    if (output_folder == raw_folder
            or raw_folder in output_folder.parents
            or output_folder in raw_folder.parents):
        raise RuntimeError(f"processed_session\n  {output_folder}\noverlaps raw recording ({role})\n"
                           f"  {raw_folder}\nRefusing to run — raw data must stay untouched.")
print("processed_session is clear of every raw recording folder — OK")

In [ ]:
# --- continuity checks: no gaps within a session, dat matches timestamps ---
for role in session_roles:
    for probe_name in probe_names:
        probe_info = sessions[role]["probes"][probe_name]
        timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy")
        intervals = np.diff(timestamps)
        expected_interval = 1.0 / probe_info["fs"]
        n_gap = int(np.count_nonzero(np.abs(intervals - expected_interval) > 0.5 * expected_interval))

        dat_bytes = (probe_info["stream_dir"] / "continuous.dat").stat().st_size
        bytes_per_sample = probe_info["n_chan"] * 2
        assert dat_bytes % bytes_per_sample == 0, \
            f"{probe_info['stream_dir']}: dat size not a multiple of n_chan*2"
        assert dat_bytes // bytes_per_sample == probe_info["n_samples"], \
            f"{probe_info['stream_dir']}: dat has {dat_bytes // bytes_per_sample} samples, " \
            f"timestamps has {probe_info['n_samples']}"

        max_deviation_us = np.abs(intervals - expected_interval).max() * 1e6
        print(f"{role:12s} {probe_name}: {probe_info['n_samples']} samp, {n_gap} gaps >50% "
              f"(max dev {max_deviation_us:.1f} us), dat size OK")
        if n_gap:
            print("  WARNING: sample gaps — session may have been paused/dropped; "
                  "concatenation assumes each session is gap-free")

In [ ]:
# --- optotagging session: detect laser pulses + estimate ADC->probe artifact lag ---
laser_channels = {name: channel for name, channel in config["adc"]["channels"].items()
                  if name.endswith("_laser")}                # {"blue_laser": 1, "red_laser": 3}

# pulse detection: threshold from the metadata, everything else from the pipeline defaults
pulse_detection_cfg = {**DEFAULT_PULSE_DETECTION,
                       "threshold_volts": config["adc"]["threshold_volts"]}
lag_estimation_cfg = DEFAULT_LAG_ESTIMATION   # probe_channel / search_window_ms / artifact_sign / max_pulses

optotagging = sessions["optotagging"]
opto_results = {}   # (probe_name, laser_name) -> {onset_times, offset_times, lag_ms, per_pulse_ms, adc_channel}
for probe_name in probe_names:
    probe_info = optotagging["probes"][probe_name]
    probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
    for laser_name, adc_channel in laser_channels.items():
        adc_volts, adc_timestamps, _ = oe_load_adc_column(
            optotagging["adc_dir"], optotagging["adc_params"], adc_channel)
        onset_times, offset_times = detect_laser_pulses(
            adc_volts, adc_timestamps,
            threshold_volts=pulse_detection_cfg["threshold_volts"],
            polarity=pulse_detection_cfg["polarity"],
            min_width_ms=pulse_detection_cfg["min_width_ms"],
            max_width_ms=pulse_detection_cfg["max_width_ms"],
            min_interpulse_ms=pulse_detection_cfg["min_interpulse_ms"])
        if len(onset_times) == 0:
            raise RuntimeError(f"{probe_name} {laser_name}: no pulses on ADC{adc_channel} "
                               "— check threshold / polarity / width")
        lag_ms, per_pulse_ms = estimate_artifact_lag(
            probe_info["stream_dir"] / "continuous.dat", probe_timestamps,
            probe_info["n_chan"], onset_times,
            probe_channel=lag_estimation_cfg["probe_channel"],
            search_window_ms=tuple(lag_estimation_cfg["search_window_ms"]),
            artifact_sign=lag_estimation_cfg["artifact_sign"],
            max_pulses=lag_estimation_cfg["max_pulses"])
        lag_mad_ms = float(np.nanmedian(np.abs(per_pulse_ms - np.nanmedian(per_pulse_ms))))
        print(f"{probe_name} {laser_name:10s}: {len(onset_times):4d} pulses, "
              f"width {np.median((offset_times - onset_times) * 1e3):.1f} ms, "
              f"lag {lag_ms:+.3f} ms (MAD {lag_mad_ms:.3f})")
        opto_results[(probe_name, laser_name)] = dict(
            onset_times=onset_times, offset_times=offset_times, lag_ms=lag_ms,
            per_pulse_ms=per_pulse_ms, adc_channel=adc_channel)

In [ ]:
# --- QC: lag histogram + pulse-triggered average with the mask window overlaid ---
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

for (probe_name, laser_name), result in opto_results.items():
    probe_info = sessions["optotagging"]["probes"][probe_name]
    fs = probe_info["fs"]
    probe_channel = lag_estimation_cfg["probe_channel"]
    mask_cfg = mask_laser_params[probe_name]
    probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
    source = np.memmap(probe_info["stream_dir"] / "continuous.dat", dtype="int16",
                       mode="r").reshape(probe_info["n_samples"], probe_info["n_chan"])

    n_before, n_after = int(0.005 * fs), int(0.015 * fs)
    snippets = []
    for onset_time in result["onset_times"][:200]:
        center = int(np.searchsorted(probe_timestamps, onset_time))
        if n_before <= center < probe_info["n_samples"] - n_after:
            snippets.append(source[center - n_before:center + n_after, probe_channel].astype("float64"))
    snippets = np.array(snippets)
    t_ms = (np.arange(-n_before, n_after) / fs) * 1e3

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    valid_lags = result["per_pulse_ms"][~np.isnan(result["per_pulse_ms"])]
    axes[0].hist(valid_lags, 40, color="0.6")
    axes[0].axvline(result["lag_ms"], color="r")
    axes[0].set(xlabel="artifact lag (ms)", ylabel="pulses",
                title=f"{probe_name} {laser_name}: lag {result['lag_ms']:+.3f} ms")
    if len(snippets):
        mean_trace = snippets.mean(0)
        axes[1].plot(t_ms, mean_trace - np.median(mean_trace), "k")
    axes[1].axvline(0, color="b", lw=0.8, label="ADC onset")
    axes[1].axvline(result["lag_ms"], color="r", lw=0.8, label="est. lag")
    axes[1].axvspan(result["lag_ms"] - mask_cfg["pre_on_ms"],
                    result["lag_ms"] + mask_cfg["post_on_ms"],
                    color="r", alpha=0.15, label="onset mask")
    axes[1].set(xlabel="time from ADC onset (ms)", ylabel="uV (rel.)",
                title=f"{probe_name} ch{probe_channel} pulse-triggered avg (n={len(snippets)})")
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

In [ ]:
# --- SpikeInterface: per session  mask -> phase shift,  then concatenate + write int16 ---
processed_session.mkdir(parents=True, exist_ok=True)

# job kwargs for write_binary_recording — set n_jobs to 1 if multiprocessing misbehaves
write_job_kwargs = dict(n_jobs=4, chunk_duration="2s", progress_bar=True)

session_boundaries = {}   # probe_name -> list of per-session {sample_start, sample_end, ...}
laser_events = []

for probe_name in probe_names:
    order           = concatenate_order[probe_name]        # e.g. ["behavior", "optotagging"]
    phase_shift_cfg = phase_shift_params[probe_name]
    mask_cfg        = mask_laser_params[probe_name]
    n_chan          = sessions[order[0]]["probes"][probe_name]["n_chan"]
    sample_shifts   = npx2_sample_shifts(n_chan)            # NP2.0 inter-sample shift, in samples

    per_session_recordings = []
    session_bounds = []
    cursor = 0
    for role in order:
        probe_info = sessions[role]["probes"][probe_name]

        # raw int16 units, memmap opened read-only by SpikeInterface
        recording = si.read_binary(probe_info["stream_dir"] / "continuous.dat",
                                   sampling_frequency=probe_info["fs"], dtype="int16",
                                   num_channels=probe_info["n_chan"])

        if role == "optotagging" and mask_cfg["apply"]:
            # laser on/off -> lag-corrected probe sample indices (this session's own clock)
            probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
            onset_frames, offset_frames = [], []
            for laser_name in laser_channels:
                result = opto_results[(probe_name, laser_name)]
                lag_seconds = result["lag_ms"] / 1000.0
                onset_frames.append(np.searchsorted(probe_timestamps, result["onset_times"] + lag_seconds))
                offset_frames.append(np.searchsorted(probe_timestamps, result["offset_times"] + lag_seconds))
            last_sample = probe_info["n_samples"] - 2
            onset_frames  = np.clip(np.concatenate(onset_frames), 1, last_sample)
            offset_frames = np.clip(np.concatenate(offset_frames), 1, last_sample)
            # remove_artifacts takes one (ms_before, ms_after) per call -> onset and offset separately
            recording = remove_artifacts(recording, [onset_frames],
                                         ms_before=mask_cfg["pre_on_ms"],
                                         ms_after=mask_cfg["post_on_ms"], mode="linear")
            recording = remove_artifacts(recording, [offset_frames],
                                         ms_before=mask_cfg["pre_off_ms"],
                                         ms_after=mask_cfg["post_off_ms"], mode="linear")
            print(f"{probe_name} {role}: masked {len(onset_frames)} onsets + {len(offset_frames)} offsets")

        if phase_shift_cfg["apply"]:
            recording = phase_shift(recording,
                                    margin_ms=phase_shift_cfg.get("margin_ms", 40.0),
                                    inter_sample_shift=sample_shifts)

        per_session_recordings.append(recording)
        session_bounds.append(dict(
            session_id=role, source_session=sessions[role]["folder"].name,
            source_dat_bytes=(probe_info["stream_dir"] / "continuous.dat").stat().st_size,
            sample_start=cursor, sample_end=cursor + probe_info["n_samples"],
            n_samples=probe_info["n_samples"]))
        cursor += probe_info["n_samples"]

    # sample-based concatenation (ignore_times=True): no wall-clock alignment across sessions
    full_recording = concatenate_recordings(per_session_recordings)
    output_dat = processed_session / probe_name / "continuous.dat"
    output_dat.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {probe_name}  ->  {output_dat}  ({full_recording.get_num_samples()} samples) ===")
    write_binary_recording(full_recording, output_dat, dtype="int16",
                           add_file_extension=False, **write_job_kwargs)
    session_boundaries[probe_name] = session_bounds

    # every masked pulse, as a sample index into the concatenated axis
    session_start = {entry["session_id"]: entry["sample_start"] for entry in session_bounds}
    if "optotagging" in order:
        probe_info = sessions["optotagging"]["probes"][probe_name]
        probe_timestamps = np.load(probe_info["stream_dir"] / "timestamps.npy").astype("float64")
        optotagging_start = session_start["optotagging"]
        for laser_name in laser_channels:
            result = opto_results[(probe_name, laser_name)]
            lag_seconds = result["lag_ms"] / 1000.0
            for onset_time, offset_time in zip(result["onset_times"], result["offset_times"]):
                onset_sample  = int(np.searchsorted(probe_timestamps, onset_time + lag_seconds))
                offset_sample = int(np.searchsorted(probe_timestamps, offset_time + lag_seconds))
                laser_events.append(dict(
                    probe=probe_name, session="optotagging", color=laser_name,
                    on_sample_concat=onset_sample + optotagging_start,
                    off_sample_concat=offset_sample + optotagging_start,
                    on_t_oe=float(onset_time), lag_ms=float(result["lag_ms"])))

print("\nconcatenation done")

In [ ]:
# --- sidecars: session boundaries, preprocessing provenance, laser events ---
boundaries_doc = {"session_name": session_name,
                  "engine": f"spikeinterface {si.__version__}",
                  "concatenate_order": concatenate_order[probe_names[0]],
                  "streams": {}}
for probe_name in probe_names:
    reference_info = sessions[session_roles[0]]["probes"][probe_name]
    boundaries_doc["streams"][probe_name] = {
        "fs": reference_info["fs"], "n_chan": reference_info["n_chan"],
        "n_samples_total": sum(entry["n_samples"] for entry in session_boundaries[probe_name]),
        "sessions": session_boundaries[probe_name]}
(processed_session / "session_boundaries.json").write_text(json.dumps(boundaries_doc, indent=2))

provenance = {"session_name": session_name, "metadata": str(metadata_path),
              "engine": f"spikeinterface {si.__version__}",
              "sessions": {role: sessions[role]["folder"].name for role in session_roles},
              "probes": {}}
for probe_name in probe_names:
    provenance["probes"][probe_name] = {
        "phase_shift": phase_shift_params[probe_name],
        "mask_laser_artifact": mask_laser_params[probe_name],
        "car": car_params.get(probe_name),
        "kilosort_params": config[probe_name]["kilosort_params"],
        "lag_ms": {laser_name: opto_results[(probe_name, laser_name)]["lag_ms"]
                   for laser_name in laser_channels},
    }
(processed_session / "preprocessing.json").write_text(json.dumps(provenance, indent=2, default=str))

if laser_events:
    with open(processed_session / "laser_events.csv", "w", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, list(laser_events[0]))
        writer.writeheader()
        writer.writerows(laser_events)

print("wrote session_boundaries.json, preprocessing.json"
      + (", laser_events.csv" if laser_events else ""))

In [9]:
# --- Kilosort channel map (.mat) per probe stream ---
for probe_name in probe_names:
    save_kilosort_channel_map(
        sessions[session_roles[0]]["probes"][probe_name]["params"],
        processed_session / probe_name, session_name,
        bad_channels=config[probe_name]["kilosort_params"]["bad_channels"],
        overwrite=True)

Saved IM-1979_2026-09-11_ProbeA.mat  (378 channels, 6 excluded)
Saved IM-1979_2026-09-11_ProbeB.mat  (380 channels, 4 excluded)


In [ ]:
# --- verification ---
probe_name = probe_names[0]
n_chan = sessions[session_roles[0]]["probes"][probe_name]["n_chan"]
output = np.memmap(processed_session / probe_name / "continuous.dat", dtype="int16",
                   mode="r").reshape(-1, n_chan)

total_samples = boundaries_doc["streams"][probe_name]["n_samples_total"]
assert output.shape[0] == total_samples, (output.shape, total_samples)
assert (processed_session / probe_name / "continuous.dat").stat().st_size == sum(
    entry["source_dat_bytes"] for entry in session_boundaries[probe_name])
print(f"{probe_name}: {output.shape[0]} samples == sum of sessions, byte count matches sources  OK")

# masked pulse: raw source vs concat output around the first laser event
if laser_events:
    event = laser_events[0]
    probe_info = sessions["optotagging"]["probes"][probe_name]
    probe_channel = lag_estimation_cfg["probe_channel"]
    source = np.memmap(probe_info["stream_dir"] / "continuous.dat", dtype="int16",
                       mode="r").reshape(probe_info["n_samples"], probe_info["n_chan"])
    optotagging_start = {entry["session_id"]: entry["sample_start"]
                         for entry in session_boundaries[probe_name]}["optotagging"]
    source_onset = event["on_sample_concat"] - optotagging_start
    half_window = int(0.006 * probe_info["fs"])

    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(np.arange(-half_window, half_window),
            source[source_onset - half_window:source_onset + half_window, probe_channel],
            "k", lw=0.8, label="raw source")
    ax.plot(np.arange(-half_window, half_window),
            output[event["on_sample_concat"] - half_window:event["on_sample_concat"] + half_window, probe_channel],
            "r", lw=0.9, label="concat output")
    ax.axvline(0, color="b", lw=0.7)
    ax.set(title=f"{event['session']} {event['color']} pulse — {probe_name} ch{probe_channel}",
           xlabel="samples from lag-corrected onset", ylabel="int16")
    ax.legend()
    plt.show()

# phase shift moved a quiet channel of the first session by < 1 sample
first_bounds = session_boundaries[probe_name][0]
first_info = sessions[session_roles[0]]["probes"][probe_name]
first_source = np.memmap(first_info["stream_dir"] / "continuous.dat", dtype="int16",
                         mode="r").reshape(first_info["n_samples"], first_info["n_chan"])
window = slice(first_info["n_samples"] // 2, first_info["n_samples"] // 2 + 16000)
source_trace = first_source[window, 5].astype("float64")
output_trace = output[first_bounds["sample_start"] + np.arange(window.start, window.stop), 5].astype("float64")
cross_corr = np.fft.irfft(np.fft.rfft(source_trace - source_trace.mean())
                          * np.conj(np.fft.rfft(output_trace - output_trace.mean())), n=len(source_trace))
peak = int(np.argmax(cross_corr))
peak = peak - len(source_trace) if peak > len(source_trace) // 2 else peak
print(f"phase-shift lag on {probe_name} ch5 (first session): {peak} sample(s)  "
      f"{'OK' if abs(peak) <= 1 else 'CHECK'}")

## Next steps

- **Channel map** — written above to `<processed_session>/<probe>/<session_name>_<ProbeX>.mat`;
  you can skip `3.1_generate_channel_map.ipynb`.
- **Kilosort** (`3.2_run_kilosort.ipynb`): point `dat_file` at
  `<processed_session>/<probe>/continuous.dat`, set `n_chan_bin` to that probe's
  channel count, set `do_CAR` from `config[<probe>]["kilosort_params"]["do_car"]`.
- **Split back per session** — spike times from the combined sort are sample
  indices into the concatenated axis. Use `session_boundaries.json` to slice them
  back: `sorting.frame_slice(sample_start, sample_end)` for the behavior session
  (first, so `sample_start = 0`). Build a behavior-scoped `analyzer.zarr` from that
  for `jdb_to_nwb` — its `add_kilosort_bombcell_spikes` checks
  `analyzer.get_num_samples() == ephys_num_samples` (the behavior session's full
  sample count), which a concatenated analyzer would fail.
- **Optotagging** — `laser_events.csv` has every pulse as a concatenated-axis
  sample index (subtract the optotagging session's `sample_start` for
  session-local indices).

### SpikeInterface-native alternative for the split-back

Instead of `concatenate_recordings` + manual `session_boundaries.json`, you can
`append_recordings(per_session_recordings)` to build a **multi-segment**
recording. `run_sorter` concatenates the segments internally, and the returned
`sorting` keeps the segments, so `sorting.select_segments([0])` gives the
behavior-session spike train directly (no frame offset). This notebook uses
`concatenate_recordings` because Kilosort's own binary input and the sidecar JSON
stay identical to the numpy version.